# Working With Files in Python

## Learning goals
By the end of the lesson, you will be able to:
- Create folders and text files.
- Read, write, and append text.
- List and find files.
- Copy, rename, move, and delete files.



## 1. Prepare a practice folder

A file stores data. A folder, also called a directory, groups files and other folders.

`Path` represents a file or folder location. `TemporaryDirectory` creates a separate workspace for this lesson. Keep the setup cell unchanged.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
import shutil

workspace = TemporaryDirectory(prefix="python_files_lesson_")
folder = Path(workspace.name)

print("Practice folder exists:", folder.exists())

Practice folder exists: True


## 2. Create a folder

Use `/` to join a folder path with a name. This builds a path; it does not create the folder.

Call `mkdir()` to create it. `exist_ok=True` avoids an error if that folder already exists.

In [2]:
documents = folder / "documents"
documents.mkdir(exist_ok=True)

print("Folder name:", documents.name)
print("Is a folder:", documents.is_dir())

Folder name: documents
Is a folder: True


## 3. Open and write and read a text file

Reading and writing files in Python begins with opening a file in the appropriate mode. The following example uses `with` and `open()` to write and read a text file’s contents and automatically close the file when the block ends. `with` creates the context in which the files is handled.

Use `open()` to open a file. Mode `"w"` creates the file or replaces its existing contents. `encoding="utf-8"` specifies the text encoding.

Be careful: writing in mode `"w"` removes any previous contents.

Alternative to `with` context manager is using `open()`/`close()` functions. In this case is responsability of the programer to explicitly call the `close()` function

In [3]:
notes = documents / "notes.txt"

with open(notes, "w", encoding="utf-8") as file:
    file.write("Python file handling\n")
    file.write("We are learning to write files.\n")

print("File exists:", notes.exists())

File exists: True


## Read a text file

Mode `"r"` opens a file for reading. `read()` returns its contents as a string. The file must already exist.

In [4]:
with open(notes, "r", encoding="utf-8") as file:
    contents = file.read()

print(contents, end="")

Python file handling
We are learning to write files.


## Add text without replacing the file

Mode `"a"` appends text at the end. Unlike `"w"`, it keeps the existing contents.

`\n` adds a new line.

In [5]:
with open(notes, "a", encoding="utf-8") as file:
    file.write("This line was added later.\n")

with open(notes, "r", encoding="utf-8") as file:
    print(file.read(), end="")

Python file handling
We are learning to write files.
This line was added later.


## Read one line at a time

Loop directly over the open file to process its lines. `print()` normally adds a newline; `end=""` avoids adding a second one here.

In [6]:
with open(notes, "r", encoding="utf-8") as file:
    for line in file:
        print(line, end="")

Python file handling
We are learning to write files.
This line was added later.


### Remember the three modes

| Mode | Purpose | Existing contents |
|---|---|---|
| `"r"` | Read | Kept |
| `"w"` | Write | Replaced |
| `"a"` | Append | Kept |

Before running a writing cell, predict whether it will replace or extend the file.

## 4. List files and folders

`iterdir()` gives the entries directly inside a folder. Use `.name` to display just the entry name.

The examples sort the names to make the display order easy to follow.

In [7]:
for entry in sorted(folder.iterdir()):
    print(entry.name)

for entry in sorted(documents.iterdir()):
    print(entry.name)

documents
notes.txt


### List only files

`is_file()` checks whether a path points to a file. `is_dir()` checks for a folder.

In [8]:
for entry in sorted(folder.iterdir()):
    print(entry.name, "is a folder:", entry.is_dir())

for entry in sorted(documents.iterdir()):
    if entry.is_file():
        print("File:", entry.name)

documents is a folder: True
File: notes.txt


## 5. Find files by name pattern

`glob("*.txt")` finds entries whose names end in `.txt` in the selected folder. The `*` matches any number of characters.

Use `rglob("*.txt")` to search inside subfolders too. Check `is_file()` when you want only files.

In [9]:
for path in sorted(documents.glob("*.txt")):
    if path.is_file():
        print("Text file:", path.name)

for path in sorted(folder.rglob("*.txt")):
    if path.is_file():
        print("Found:", path.relative_to(folder))

Text file: notes.txt
Found: documents/notes.txt


## 6. Check file properties

A path has a name and a suffix, such as `.txt`. `stat()` provides file information; `st_size` is the size in bytes.

In [10]:
print("Name:", notes.name)
print("Suffix:", notes.suffix)
print("Size in bytes:", notes.stat().st_size)

Name: notes.txt
Suffix: .txt
Size in bytes: 80


## 7. Copy a file

`shutil.copy()` creates a copy at the destination. The original stays in place.

An existing destination file can be overwritten. Use the practice folder for this example.

In [11]:
copied_file = documents / "notes_copy.txt"
shutil.copy(notes, copied_file)

print("Original exists:", notes.exists())
print("Copy exists:", copied_file.exists())

Original exists: True
Copy exists: True


## 8. Rename a file

`rename()` changes the path of a file. Here, the file stays in the same folder but receives a new name.

The old `Path` variable still describes the old name. Store the new path to use it in later operations. Use a destination name that does not already exist.

In [12]:
renamed_file = documents / "backup.txt"
copied_file.rename(renamed_file)

print("Old name exists:", copied_file.exists())
print("New name exists:", renamed_file.exists())

Old name exists: False
New name exists: True


## 9. Move a file

`shutil.move()` moves a file to another location. Here, the backup leaves the documents folder and goes into the archive folder.

In [13]:
archive_folder = folder / "archive"
archive_folder.mkdir(exist_ok=True)
moved_file = archive_folder / "backup.txt"

shutil.move(str(renamed_file), str(moved_file))

print("Old location exists:", renamed_file.exists())
print("New location exists:", moved_file.exists())

Old location exists: False
New location exists: True


## 10. Handle a missing file

Opening a file that does not exist raises `FileNotFoundError`. Use `try` and `except` to handle this situation without stopping the program.

In [14]:
missing_file = folder / "missing.txt"

try:
    with open(missing_file, "r", encoding="utf-8") as file:
        print(file.read())
except FileNotFoundError:
    print("The file was not found.")

The file was not found.


## 11. Delete a practice file

`unlink()` deletes a file. Deletion does not send it to the recycle bin.

Only delete the backup created during this lesson.

In [15]:
moved_file.unlink()
print("Backup still exists:", moved_file.exists())

Backup still exists: False


### Delete an empty folder

`rmdir()` removes a folder only if it is empty. The archive folder is now empty because its backup file was deleted.

In [16]:
archive_folder.rmdir()
print("Archive folder still exists:", archive_folder.exists())

Archive folder still exists: False


## Optional: Create and read a ZIP archive

A ZIP archive groups files in one compressed file. Use `"w"` to create an archive. `arcname` sets the name stored inside it.

Use `"r"` to inspect the archive. This example extracts only the archive just created in the practice folder. Do not extract an unknown archive without checking its contents.

In [17]:
from zipfile import ZipFile

zip_path = folder / "notes.zip"

with ZipFile(zip_path, "w") as archive:
    archive.write(notes, arcname="notes.txt")

with ZipFile(zip_path, "r") as archive:
    print("Files in the archive:", archive.namelist())
    archive.extractall(folder / "extracted")

print("Extracted file exists:", (folder / "extracted" / "notes.txt").exists())

Files in the archive: ['notes.txt']
Extracted file exists: True


## Practice

Use only the lesson’s practice folder.

1. Create a file named `practice.txt` inside `documents`.
2. Write two lines to it.
3. Append a third line.
4. Read and display the contents.
5. List all `.txt` files in `documents`.

Before coding, choose the mode needed for each file operation.

In [18]:
# Write your solution here.

## Finish: clean up the practice folder

Run this cell after completing the exercises. It removes the temporary workspace and everything created inside it.

To repeat the lesson, start again from the setup cell.

In [19]:
workspace.cleanup()
print("Practice folder still exists:", folder.exists())

Practice folder still exists: False
